# Flickr30k Sequence-KD negative-result closeout

Ghi nhận và khóa hai kết quả âm T1→S1 và T2→S2 trên Validation. Notebook
không huấn luyện lại, không thay đổi ngưỡng và không đọc Held-out Test.

## Kaggle input bắt buộc — đúng 2 input

1. Output `F30K_T1S1_SeqKD_Aggregate`.
2. Output `F30K_T2S2_SeqKD_Aggregate`.

Runtime: CPU/Accelerator None, Internet Off, dự kiến dưới 1 phút. Có thể chạy
song song với `F30K_T3S3_Final_Lock` và các seed DenseMap.

## Cell 1 — Verify, close and retain negative results

In [1]:
EXPERIMENT_ID = "F30K_SeqKD_Closeout"


import csv
import hashlib
import json
import math
import os
import statistics
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

def verify_manifest(root):
    manifest_path = Path(root) / "MANIFEST.json"
    manifest = read_json(manifest_path)
    assert manifest["status"] == "complete"
    assert manifest["test_set_status"] == "sealed"
    assert manifest["test_set_used"] is False
    for artifact in manifest["artifacts"]:
        path = Path(root) / artifact["relative_path"]
        assert path.is_file(), path
        assert path.stat().st_size == int(artifact["size_bytes"])
        assert sha256(path) == artifact["sha256"]
    return manifest_path, manifest

branches = [
    ("T1S1_SeqKD", "Sequence KD"),
    ("T2S2_SeqKD", "Sequence KD"),
]
rows = []
source_hashes = []
for pair_id, method in branches:
    experiment = f"F30K_{pair_id}_Aggregate"
    result_path = unique_named(experiment + "_RESULT.json")
    result = read_json(result_path)
    manifest_path, _ = verify_manifest(result_path.parent)
    assert result["status"] == "complete"
    assert result["experiment_id"] == experiment
    assert result["pair_id"] == pair_id
    assert result["method"] == method
    assert result["seeds"] == [42, 123, 2026]
    assert result["run_count"] == 3
    assert result["consistent_positive_delta"] is False
    assert result["consistent_negative_delta"] is True
    assert result["decision"] == "STOP_VALIDATION_NO_GAIN"
    assert result["next_phase_authorized"] is None
    assert result["test_set_status"] == "sealed"
    assert result["test_set_used"] is False
    combined = result["combined_paired_bootstrap"]
    assert float(combined["ci95_upper"]) < 0
    rows.append({
        "pair_id": pair_id,
        "method": method,
        "seed_count": 3,
        "mean_delta_CIDEr_D": result["summary"]["CIDEr_D"]["delta_mean"],
        "sample_sd_delta_CIDEr_D": result["summary"]["CIDEr_D"][
            "delta_sample_sd"
        ],
        "combined_ci95_lower": combined["ci95_lower"],
        "combined_ci95_upper": combined["ci95_upper"],
        "bootstrap_positive_probability": combined[
            "bootstrap_positive_probability"
        ],
        "decision": result["decision"],
    })
    source_hashes.append({
        "experiment_id": experiment,
        "result_sha256": sha256(result_path),
        "manifest_sha256": sha256(manifest_path),
    })

closeout = {
    "schema_version": "f30k_negative_sequence_kd_closeout_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete",
    "experiment_id": EXPERIMENT_ID,
    "closed_branches": rows,
    "source_hashes": source_hashes,
    "scientific_conclusion": (
        "Sequence KD decreased Validation CIDEr-D consistently for both "
        "T1->S1 and T2->S2 across seeds 42, 123, and 2026."
    ),
    "retraining_or_test_authorized": False,
    "retain_as_negative_results": True,
    "next_phase_authorized": None,
    "test_set_status": "sealed",
    "test_set_used": False,
}
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
csv_path = WORK_ROOT / "closed_branches.csv"
report_path = WORK_ROOT / "REPORT.md"
write_json(result_path, closeout)
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
lines = [
    "# Flickr30k Sequence-KD negative-result closeout",
    "",
    "| Pair | Mean Δ CIDEr-D | CI95 | Decision |",
    "|---|---:|---:|---|",
]
for row in rows:
    lines.append(
        f"| {row['pair_id']} | {row['mean_delta_CIDEr_D']:+.6f} | "
        f"[{row['combined_ci95_lower']:+.6f}; "
        f"{row['combined_ci95_upper']:+.6f}] | "
        f"`{row['decision']}` |"
    )
lines.extend(["", "- Held-out Test: `SEALED / NOT USED`"])
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

artifacts = []
for path in [result_path, csv_path, report_path]:
    artifacts.append({
        "relative_path": path.name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })
write_json(WORK_ROOT / "MANIFEST.json", {
    "schema_version": "f30k_sequence_kd_closeout_manifest_v1",
    "experiment_id": EXPERIMENT_ID,
    "status": "complete",
    "artifacts": artifacts,
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("Status                 : COMPLETE")
for row in rows:
    print(row["pair_id"], "mean delta:",
          f"{float(row['mean_delta_CIDEr_D']):+.6f}",
          "CI95:", row["combined_ci95_lower"], row["combined_ci95_upper"])
print("Decision               : STOP BOTH SEQUENCE-KD BRANCHES")
print("Held-out Test          : SEALED / NOT USED")

Status                 : COMPLETE
T1S1_SeqKD mean delta: -0.092395 CI95: -0.11654129643171586 -0.06886039731940308
T2S2_SeqKD mean delta: -0.073892 CI95: -0.09694771290049978 -0.05049252932182689
Decision               : STOP BOTH SEQUENCE-KD BRANCHES
Held-out Test          : SEALED / NOT USED
